In [0]:
%run ./00_config

# Pipeline tests
Read-only checks: proves the pipeline is incremental, handles duplicates and corrections, is idempotent, and is replayable.
Run step by step, following the instructions between cells. Destructive cells live in `99_reset`.

In [0]:
from pyspark.sql import functions as F

TABLES = ["bronze_trades", "bronze_prices", "silver_trades", "silver_prices", "silver_quarantine",
          "gold_trade_ledger", "gold_positions", "gold_portfolio_valuation", "gold_client_aum"]

def snapshot():
    return {t: spark.table(t).count() for t in TABLES}

def compare(before, after):
    for t in TABLES:
        print(f"{t:26} {before[t]:>7} → {after[t]:>7}   (+{after[t] - before[t]})")

def fingerprint():
    p = spark.table("gold_positions").agg(F.sum("quantity"), F.sum("cost_basis"),
                                          F.sum("realized_pnl_total")).first()
    aum = spark.table("gold_client_aum").agg(F.sum("aum")).first()[0]
    return {"total_quantity": p[0], "total_cost_basis": p[1], "total_realized": p[2], "total_aum": aum}

def pairs_updated_last_run():
    latest = spark.table("gold_positions").agg(F.max("_updated_at")).first()[0]
    return spark.table("gold_positions").filter(F.col("_updated_at") == latest).count()


def oversells():
    return tbl("gold_positions").agg(F.sum("oversell_count")).first()[0]

%md
## Step 1: Baseline
Run after the backfill has gone through the Job, before landing any test day.

In [0]:
base = snapshot()
print(base)
print("total pairs:", tbl("gold_positions").count(), "| oversells:", oversells())

%md
## Step 2: Day 1 (new trades + duplicates)
In `01_source_generator`, run `land_day(test_day1)`. Then run the Job.

In [0]:
after_day1 = snapshot()
compare(base, after_day1)
print("pairs recalculated:", pairs_updated_last_run(), "| oversells:", oversells())

%md
## Step 3a: Before day 2, photograph the trade that will be corrected
Set `CORR_ID` to the correction ID printed by `01_source_generator`. Run this **before** landing day 2.

In [0]:
CORR_ID = "TRD-006366"   

corr = tbl("silver_trades").filter(F.col("trade_id") == CORR_ID).first()
pair = (F.col("client_id") == corr.client_id) & (F.col("asset_id") == corr.asset_id)

before_ledger = tbl("gold_trade_ledger").filter(F.col("trade_id") == CORR_ID).first()
before_pair = tbl("gold_positions").filter(pair).first()
print("ledger before:", before_ledger.unit_price, before_ledger.avg_cost_after)
print("position before:", before_pair.avg_cost, before_pair.cost_basis)

%md
## Step 3b: Day 2 (correction)
In `01_source_generator`, run `land_day(test_day2)`. Then run the Job.

In [0]:
after_day2 = snapshot()
compare(after_day1, after_day2)
print("pairs recalculated:", pairs_updated_last_run(), "| oversells:", oversells())

display(tbl("bronze_trades").filter(F.col("trade_id") == CORR_ID)
        .select("trade_id", "unit_price", "updated_at", "_source_file"))      # original row(s) + correction
display(tbl("silver_trades").filter(F.col("trade_id") == CORR_ID)
        .select("trade_id", "unit_price", "updated_at"))                      # 1 row, corrected price

after_ledger = tbl("gold_trade_ledger").filter(F.col("trade_id") == CORR_ID).first()
after_pair = tbl("gold_positions").filter(pair).first()
print("ledger:   price", before_ledger.unit_price, "→", after_ledger.unit_price,
      "| avg cost", before_ledger.avg_cost_after, "→", after_ledger.avg_cost_after)
print("position: avg cost", before_pair.avg_cost, "→", after_pair.avg_cost,
      "| cost basis", before_pair.cost_basis, "→", after_pair.cost_basis)

In [0]:
%sql
-- expect numTargetRowsUpdated = 1 (the correction) and numTargetRowsInserted = day 2's new trades
SELECT version, operation, operationMetrics.numTargetRowsInserted, operationMetrics.numTargetRowsUpdated
FROM (DESCRIBE HISTORY silver_trades) LIMIT 1;

%md
## Step 4: Rerun with no new data (idempotency)
Run the Job again without landing anything.

In [0]:
fp_before_rerun = fingerprint()

In [0]:
after_rerun = snapshot()
compare(after_day2, after_rerun)
print("fingerprint unchanged:", fingerprint() == fp_before_rerun)

## Step 5: Replay (rebuild Silver + Gold from Bronze)
1. Run the cell below to record the fingerprint.
2. In `99_reset`, run the **Replay** cell.
3. Run the Job.
4. Run the comparison cell.

In [0]:
fp_before_replay = fingerprint()
print(fp_before_replay)

In [0]:
fp_after_replay = fingerprint()
print("before:", fp_before_replay)
print("after: ", fp_after_replay)
print("IDENTICAL" if fp_before_replay == fp_after_replay else "DIFFERENT")

Day 1: new trades / duplicates	26 trades landed (29 rows in Bronze), 3 duplicates removed in Silver
Day 1: pairs recalculated	26 of 1,194 total pairs (about 2%)
Day 1: new trades / duplicates	26 trades landed (29 rows in Bronze), 3 duplicates removed in Silver
Day 1: pairs recalculated	26 of 1,194 total pairs (about 2%)